In [1]:
%load_ext autoreload
%autoreload 2

import os
# Use the async CUDA allocator — avoids fragmentation when large tensors
# (full-volume label arrays ~8 GiB each) compete with many small allocations.
# Must be set before TF initialises.
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import sys, glob, gc
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path
from hydra import compose, initialize
from omegaconf import OmegaConf

# Patchwork repo
sys.path.insert(0, "/nfs/norasys/notebooks/camaret/repos/patchwork")
import patchwork

# GPU memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

with initialize(config_path="dev/whole_body_benchmark/configs", version_base=None):
    cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

I0000 00:00:1776845472.122221      79 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/



In [2]:
# list subjects in the current nnUNet dataset
nnunet_ds_name = "Dataset004_Nako_oppscreen"

train_dir = Path(cfg.paths.nnUNet_dir) / f"nnUNet_raw/{nnunet_ds_name}/imagesTr"
val_dir   = Path(cfg.paths.nnUNet_dir) / f"nnUNet_raw/{nnunet_ds_name}/imagesTs"

train_imgs   = train_dir.glob("*.nii.gz")
train_ids    = [f.stem.split("_")[0] for f in train_imgs]
subjects_tr  = list(set(train_ids))

val_imgs     = val_dir.glob("*.nii.gz")
val_ids      = [f.stem.split("_")[0] for f in val_imgs]
subjects_val = list(set(val_ids))

print(f"Train: {len(subjects_tr)}  Val: {len(subjects_val)}")

Train: 80  Val: 20
